# Merge Tile-Level Masks → Slide-Level Mask

把所有 tile level 的 edited mask 拼成一张完整 slide mask。

流程：
1. 发现所有 edited tile mask，确定 canvas 尺寸  
2. 逐 tile paste，每个 tile 的 ID 加全局偏移（避免冲突）  
3. **Boundary stitching**：检查相邻 tile 边界，用 Union-Find 合并跨边界的同一 instance  
4. 重新紧凑编号 → 输出 uint32 `.npy` + companion CSV  
5. 可视化

In [1]:
import os
import re
import glob
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle

## Config

In [2]:
MASK_DIR   = "data/Visium_HD_Human_Colon_Cancer_P2/maskann_size-512/sam2_merged_masks"
TILE_SIZE  = 512           # pixels per tile (square)
OUT_MASK   = os.path.join(MASK_DIR, "slide_merged_mask.npy")
OUT_CSV    = os.path.join(MASK_DIR, "slide_merged_mask_info.csv")

## Step 1 — Discover Tiles

In [3]:
def discover_edited_tiles(mask_dir: str) -> list[dict]:
    """
    扫描 mask_dir，返回所有 edited mask 的信息列表。
    每条记录：{ prefix, row, col, path }
    """
    files = sorted(glob.glob(os.path.join(mask_dir, "*_merged_mask_edited.npy")))
    tiles = []
    for f in files:
        m = re.search(r"r(\d+)_c(\d+)", os.path.basename(f))
        if m is None:
            continue
        r, c = int(m.group(1)), int(m.group(2))
        tiles.append(dict(prefix=f"r{r}_c{c}", row=r, col=c, path=f))
    return tiles


tiles = discover_edited_tiles(MASK_DIR)

all_rows = sorted(set(t["row"] for t in tiles))
all_cols = sorted(set(t["col"] for t in tiles))
canvas_H = max(all_rows) + TILE_SIZE
canvas_W = max(all_cols) + TILE_SIZE

print(f"Found {len(tiles)} edited tiles")
print(f"Row offsets : {all_rows}")
print(f"Col offsets : {all_cols}")
print(f"Canvas size : {canvas_H} × {canvas_W}")

Found 33 edited tiles
Row offsets : [0, 512, 1024, 1536, 2048, 2560]
Col offsets : [256, 768, 1280, 1792, 2304, 2816]
Canvas size : 3072 × 3328


## Step 2 — Paste Tiles with Global ID Offset

In [4]:
canvas = np.zeros((canvas_H, canvas_W), dtype=np.uint32)

# tile_id_map[prefix] = (global_id_offset, local_max_id)
# global_id = local_id + offset  (for local_id > 0)
tile_meta = {}   # prefix -> {offset, local_max, row, col}
global_offset = 0

for t in tiles:
    local_mask = np.load(t["path"])  # uint32 or int32, shape (512,512)
    local_mask = local_mask.astype(np.uint32)

    local_max = int(local_mask.max())
    r0, c0 = t["row"], t["col"]
    r1, c1 = r0 + TILE_SIZE, c0 + TILE_SIZE

    # shift non-zero IDs by global_offset
    shifted = np.where(local_mask > 0, local_mask + global_offset, 0).astype(np.uint32)
    canvas[r0:r1, c0:c1] = shifted

    tile_meta[t["prefix"]] = dict(
        row=r0, col=c0,
        offset=global_offset,
        local_max=local_max,
    )
    global_offset += local_max

total_raw_ids = global_offset
print(f"Raw global IDs before stitching: {total_raw_ids}")
print(f"Non-zero pixels: {(canvas > 0).sum():,}")

Raw global IDs before stitching: 5002
Non-zero pixels: 4,376,672


## Step 3 — Boundary Stitching (Union-Find)

相邻 tile 的交界处：同一个细胞被切成两个局部 instance。
- 检查 top tile 最后一行 vs bottom tile 第一行（同列对齐）
- 检查 left tile 最后一列 vs right tile 第一列（同行对齐）
- 凡是在边界像素上重合（同位置都非零）的两个 global ID → Union

In [ ]:
class UnionFind:
    """路径压缩 + 按秩合并的 Union-Find。"""
    def __init__(self, n: int):
        self.parent = np.arange(n, dtype=np.uint32)
        self.rank   = np.zeros(n, dtype=np.uint8)

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]  # path compression
            x = self.parent[x]
        return int(x)

    def union(self, a: int, b: int):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


# ID 0 = background；有效 ID 范围 [1, total_raw_ids]
uf = UnionFind(total_raw_ids + 1)

# build lookup: (row, col) -> tile  for fast adjacency detection
tile_by_rc = {(t["row"], t["col"]): t for t in tiles}

merge_count = 0

for t in tiles:
    r0, c0 = t["row"], t["col"]

    # --- Vertical boundary: this tile (top) vs tile directly below ---
    neighbor_below = tile_by_rc.get((r0 + TILE_SIZE, c0))
    if neighbor_below is not None:
        boundary_row = r0 + TILE_SIZE          # first row of bottom tile
        strip_top    = canvas[boundary_row - 1, c0 : c0 + TILE_SIZE]  # last row of top tile
        strip_bot    = canvas[boundary_row,     c0 : c0 + TILE_SIZE]  # first row of bottom tile
        mask = (strip_top > 0) & (strip_bot > 0)
        pairs = np.unique(
            np.stack([strip_top[mask], strip_bot[mask]], axis=1), axis=0
        )
        for a, b in pairs:
            uf.union(int(a), int(b))
            merge_count += 1

    # --- Horizontal boundary: this tile (left) vs tile to the right ---
    neighbor_right = tile_by_rc.get((r0, c0 + TILE_SIZE))
    if neighbor_right is not None:
        boundary_col = c0 + TILE_SIZE
        strip_left   = canvas[r0 : r0 + TILE_SIZE, boundary_col - 1]  # last col of left tile
        strip_right  = canvas[r0 : r0 + TILE_SIZE, boundary_col    ]  # first col of right tile
        mask = (strip_left > 0) & (strip_right > 0)
        pairs = np.unique(
            np.stack([strip_left[mask], strip_right[mask]], axis=1), axis=0
        )
        for a, b in pairs:
            uf.union(int(a), int(b))
            merge_count += 1

print(f"Boundary instance pairs merged: {merge_count}")

Boundary instance pairs merged: 583


: 

## Step 4 — Relabel Canvas (Compact IDs)

In [ ]:
# 把每个 raw global ID 映射到它的 UF root
# 再把所有 root 紧凑重编号为 1, 2, 3, ...

# Step 4a: build root map for all IDs that actually appear in canvas
raw_ids_in_canvas = np.unique(canvas)
raw_ids_in_canvas = raw_ids_in_canvas[raw_ids_in_canvas > 0]

# find root for each raw ID
roots = np.array([uf.find(int(x)) for x in raw_ids_in_canvas], dtype=np.uint32)

# assign compact IDs to unique roots
unique_roots = np.unique(roots)
root_to_compact = {int(r): i + 1 for i, r in enumerate(unique_roots)}

# build full relabel lookup table (size = total_raw_ids+1)
lut = np.zeros(total_raw_ids + 1, dtype=np.uint32)
for raw_id, root in zip(raw_ids_in_canvas, roots):
    lut[raw_id] = root_to_compact[int(root)]

# apply LUT to canvas
slide_mask = lut[canvas]   # fancy indexing: canvas values → compact IDs

n_instances = len(unique_roots)
print(f"Instances before stitching : {len(raw_ids_in_canvas):,}")
print(f"Instances after  stitching : {n_instances:,}")
print(f"Instances merged across boundaries: {len(raw_ids_in_canvas) - n_instances:,}")
print(f"slide_mask shape: {slide_mask.shape}, dtype: {slide_mask.dtype}")

## Step 5 — Build Global Instance Table

In [ ]:
# For each compact global ID: area, global bbox, and which tiles it came from

# Build reverse map: compact_id -> list of raw_ids
from collections import defaultdict

compact_to_raws = defaultdict(list)
for raw_id, cid in zip(raw_ids_in_canvas, [lut[x] for x in raw_ids_in_canvas]):
    compact_to_raws[int(cid)].append(int(raw_id))

# raw_id -> (prefix, orig_local_id)
raw_to_tile = {}
for prefix, meta in tile_meta.items():
    off = meta["offset"]
    for local_id in range(1, meta["local_max"] + 1):
        raw_to_tile[off + local_id] = (prefix, local_id)

rows = []
for cid in range(1, n_instances + 1):
    px = np.where(slide_mask == cid)
    area = len(px[0])
    y0, y1 = int(px[0].min()), int(px[0].max())
    x0, x1 = int(px[1].min()), int(px[1].max())

    raw_ids = compact_to_raws[cid]
    tile_sources = list(set(raw_to_tile[r][0] for r in raw_ids if r in raw_to_tile))
    is_boundary  = len(tile_sources) > 1

    rows.append(dict(
        id           = cid,
        area         = area,
        bbox_y       = y0,
        bbox_x       = x0,
        bbox_h       = y1 - y0 + 1,
        bbox_w       = x1 - x0 + 1,
        tile_sources = "|".join(sorted(tile_sources)),
        is_boundary  = is_boundary,
    ))

df = pd.DataFrame(rows)
print(df.shape)
print(f"Boundary-spanning instances: {df['is_boundary'].sum()}")
df.head()

## Step 6 — Save

In [ ]:
np.save(OUT_MASK, slide_mask)
df.to_csv(OUT_CSV, index=False)

mask_mb = os.path.getsize(OUT_MASK) / 1e6
print(f"Saved mask  → {OUT_MASK}  ({mask_mb:.1f} MB)")
print(f"Saved table → {OUT_CSV}  ({len(df)} rows)")

## Step 7 — Visualization

### 7a. Full Slide Overview

In [ ]:
def random_label_cmap(n_labels: int, seed: int = 42) -> mcolors.ListedColormap:
    """随机颜色 colormap，0 = 黑色背景。"""
    rng = np.random.default_rng(seed)
    colors = rng.random((n_labels + 1, 3))
    colors[0] = 0.0  # background = black
    return mcolors.ListedColormap(colors)


# Downsample for display
DISPLAY_SCALE = 4   # show every Nth pixel
thumb = slide_mask[::DISPLAY_SCALE, ::DISPLAY_SCALE]

# Remap to small indices for colormap (just for display)
display_ids = np.unique(thumb)
display_lut = np.zeros(slide_mask.max() + 1, dtype=np.int32)
for i, v in enumerate(display_ids):
    display_lut[v] = i
thumb_remapped = display_lut[thumb]

cmap = random_label_cmap(len(display_ids))

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(thumb_remapped, cmap=cmap, interpolation="nearest",
          vmin=0, vmax=len(display_ids))

# Draw tile grid
for r in all_rows:
    ax.axhline(r / DISPLAY_SCALE, color="white", linewidth=0.4, alpha=0.5)
for c in all_cols:
    ax.axvline(c / DISPLAY_SCALE, color="white", linewidth=0.4, alpha=0.5)

ax.set_title(f"Slide Mask — {n_instances:,} instances  ({canvas_H}×{canvas_W} px)", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()

### 7b. Boundary Stitching Before vs After

选一个有较多 boundary merge 的边界，对比 stitching 前后的效果。

In [ ]:
# Find the boundary with the most merged pairs
# Re-run boundary detection to find best example
best_boundary = None
best_count = 0
best_info  = {}

for t in tiles:
    r0, c0 = t["row"], t["col"]

    neighbor_below = tile_by_rc.get((r0 + TILE_SIZE, c0))
    if neighbor_below:
        boundary_row = r0 + TILE_SIZE
        s1 = canvas[boundary_row - 1, c0 : c0 + TILE_SIZE]
        s2 = canvas[boundary_row,     c0 : c0 + TILE_SIZE]
        n  = int(((s1 > 0) & (s2 > 0)).sum())
        if n > best_count:
            best_count = n
            best_boundary = ("H", boundary_row, c0, c0 + TILE_SIZE)
            best_info = dict(t1=t["prefix"], t2=neighbor_below["prefix"],
                             direction="horizontal", row=boundary_row, n_px=n)

    neighbor_right = tile_by_rc.get((r0, c0 + TILE_SIZE))
    if neighbor_right:
        boundary_col = c0 + TILE_SIZE
        s1 = canvas[r0 : r0 + TILE_SIZE, boundary_col - 1]
        s2 = canvas[r0 : r0 + TILE_SIZE, boundary_col    ]
        n  = int(((s1 > 0) & (s2 > 0)).sum())
        if n > best_count:
            best_count = n
            best_boundary = ("V", r0, r0 + TILE_SIZE, boundary_col)
            best_info = dict(t1=t["prefix"], t2=neighbor_right["prefix"],
                             direction="vertical", col=boundary_col, n_px=n)

print(f"Best boundary: {best_info}")

In [ ]:
# Crop a window around the best boundary for visualization
HALF = 80   # pixels on each side of boundary

if best_boundary[0] == "H":
    _, brow, c_start, c_end = best_boundary
    r_lo = max(0, brow - HALF)
    r_hi = min(canvas_H, brow + HALF)
    crop_before = canvas    [r_lo:r_hi, c_start:c_end]
    crop_after  = slide_mask[r_lo:r_hi, c_start:c_end]
    boundary_in_crop = brow - r_lo
    axis_for_line = "h"
else:
    _, r_start, r_end, bcol = best_boundary
    c_lo = max(0, bcol - HALF)
    c_hi = min(canvas_W, bcol + HALF)
    crop_before = canvas    [r_start:r_end, c_lo:c_hi]
    crop_after  = slide_mask[r_start:r_end, c_lo:c_hi]
    boundary_in_crop = bcol - c_lo
    axis_for_line = "v"


def remap_for_display(arr, seed=42):
    ids = np.unique(arr)
    lut = np.zeros(int(arr.max()) + 1, dtype=np.int32)
    for i, v in enumerate(ids):
        lut[v] = i
    remapped = lut[arr]
    cmap = random_label_cmap(len(ids), seed=seed)
    return remapped, cmap, len(ids)


rb, cb, nb = remap_for_display(crop_before)
ra, ca, na = remap_for_display(crop_after)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
titles = ["Before stitching", "After stitching"]
for ax, data, cmap_, n_, title in zip(axes,
                                       [rb, ra], [cb, ca], [nb, na], titles):
    ax.imshow(data, cmap=cmap_, interpolation="nearest", vmin=0, vmax=n_)
    if axis_for_line == "h":
        ax.axhline(boundary_in_crop - 0.5, color="red", linewidth=1.5, linestyle="--")
        ax.axhline(boundary_in_crop + 0.5, color="red", linewidth=1.5, linestyle="--")
    else:
        ax.axvline(boundary_in_crop - 0.5, color="red", linewidth=1.5, linestyle="--")
        ax.axvline(boundary_in_crop + 0.5, color="red", linewidth=1.5, linestyle="--")
    ax.set_title(title, fontsize=12)
    ax.axis("off")

t1, t2 = best_info.get("t1", ""), best_info.get("t2", "")
fig.suptitle(f"Boundary: {t1} / {t2}  ({best_info['n_px']} touching pixels)",
             fontsize=11)
plt.tight_layout()
plt.show()

### 7c. Instance Size Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(df["area"], bins=80, color="steelblue", edgecolor="none")
ax.set_xlabel("Instance area (pixels)")
ax.set_ylabel("Count")
ax.set_title("Area distribution (all instances)")
ax.set_yscale("log")

ax = axes[1]
boundary_areas = df[df["is_boundary"]]["area"]
non_boundary_areas = df[~df["is_boundary"]]["area"]
ax.hist(non_boundary_areas, bins=60, alpha=0.7, label="within tile", color="steelblue")
ax.hist(boundary_areas,     bins=60, alpha=0.7, label="boundary-spanning", color="tomato")
ax.set_xlabel("Instance area (pixels)")
ax.set_ylabel("Count")
ax.set_title(f"Boundary-spanning instances (n={len(boundary_areas)})")
ax.set_yscale("log")
ax.legend()

plt.tight_layout()
plt.show()

print(df[["area", "is_boundary"]].groupby("is_boundary").agg(["count", "mean", "median"]).round(1))